In [11]:
# Step 1: Load Dataset
# Description: Import necessary libraries and read your EPC dataset into a Pandas DataFrame.
import pandas as pd
import ast  # for converting string representations of lists/dicts

file_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_window_dimensions_added.csv"
df = pd.read_csv(file_path)

# Optional: preview the first rows
df.head()

,BUILDING_REFERENCE_NUMBER,OSG_REFERENCE_NUMBER,ADDRESS1,ADDRESS2,ADDRESS3,POSTCODE,LATITUDE,LONGITUDE,INSPECTION_DATE,TYPE_OF_ASSESSMENT,...,AGE_BAND,BUILDING_HEIGHT,EXPOSED_PERIMETER,WALL_AREA,TYPICAL_WINDOW_AREA,WINDOW_AREA,WWR,SEMI_DETACHED_2F_WINDOW_AREA_SINGLE,SemiDetached_2F_Window_H,SemiDetached_2F_Window_W
0,1001829067,122009933.0,19 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.055692,-4.223337,9/29/25,"RdSAP, existing dwelling",...,E,4.8,13.747727,65.989090,12.5358,12.5358,0.189968,1.790829,1.092651,1.638976
1,1001951858,122010025.0,GLENVIEW,FINTRY,GLASGOW,G63 0XJ,56.052824,-4.220640,9/19/25,"RdSAP, existing dwelling",...,G,4.8,32.403703,155.537777,22.3276,22.3276,0.143551,3.189657,1.458231,2.187347
2,1001829071,122009868.0,22 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.055503,-4.223691,7/29/25,"RdSAP, existing dwelling",...,D,4.8,NaN,NaN,14.3142,14.3142,NaN,2.044886,1.167586,1.751379
3,1234761001,122009968.0,1 MENZIES TERRACE,FINTRY,GLASGOW,G63 0YJ,56.058427,-4.224838,7/22/25,"RdSAP, existing dwelling",...,E,4.8,32.532035,156.153770,23.0673,23.0673,0.147722,3.295329,1.482190,2.223284
4,1001991633,122009892.0,50 MAIN STREET,FINTRY,GLASGOW,G63 0XF,56.054738,-4.228307,7/17/25,"RdSAP, existing dwelling",...,G,2.4,NaN,NaN,19.3444,19.3444,NaN,2.763486,1.357322,2.035983


In [12]:
# Step 2 (Updated): Define mappings
# Description: Updated main heating mapping to include all main heating types. Secondary heating
# and energy rating mappings remain the same.

main_heat_mapping = {
    'Air source heat pump, radiators, electric': 'air_source_heat_pump_radiators_electric',
    'Air source heat pump, underfloor, electric': 'air_source_heat_pump_underfloor_electric',
    'Boiler and radiators, LPG': 'boiler_radiators_lpg',
    'Boiler and radiators, dual fuel (mineral and wood)': 'boiler_radiators_dual_fuel_mineral_wood',
    'Boiler and radiators, electric': 'boiler_radiators_electric',
    'Boiler and radiators, mains gas': 'boiler_radiators_mains_gas',
    'Boiler and radiators, oil': 'boiler_radiators_oil',
    'Boiler and radiators, wood logs': 'boiler_radiators_wood_logs',
    'Boiler and radiators, wood pellets': 'boiler_radiators_wood_pellets',
    'Community scheme': 'community_scheme',
    'Electric storage heaters': 'electric_storage_heaters',
    'Ground source heat pump, radiators, electric': 'ground_source_heat_pump_radiators_electric',
    'Ground source heat pump, underfloor, electric': 'ground_source_heat_pump_underfloor_electric',
    'No system present: electric heaters assumed': 'no_system_present_electric_heaters_assumed',
    'Room heaters, dual fuel (mineral and wood)': 'room_heaters_dual_fuel_mineral_wood',
    'Room heaters, electric': 'room_heaters_electric'
}

secondary_heat_mapping = {
    'None': 'none',
    'Portable electric heaters (assumed)': 'portable_electric_heaters',
    'Room heaters, coal': 'room_heaters_coal',
    'Room heaters, dual fuel (mineral and wood)': 'room_heaters_dual_fuel',
    'Room heaters, electric': 'room_heaters_electric',
    'Room heaters, wood logs': 'room_heaters_wood_logs',
    'Room heaters, wood pellets': 'room_heaters_wood_pellets'
}

energy_rating_mapping = {
    'Very Poor': 'very_poor',
    'Poor': 'poor',
    'Average': 'average',
    'Good': 'good',
    'Very Good': 'very_good'
}

In [13]:
# Step 3: Helper function to extract first description from list column
# Description: Convert string representation of list of dicts into Python objects and extract main fields.

def extract_main_heat_code(main_col):
    try:
        # Convert string to list of dicts
        main_list = ast.literal_eval(main_col)
        desc = main_list[0]['description']
        energy = main_list[0]['energy_rating']
        
        main_code = main_heat_mapping.get(desc, 'unknown_main')
        energy_code = energy_rating_mapping.get(energy, 'unknown_energy')
        
        return f"{main_code}_{energy_code}"
    except:
        return 'unknown_main_unknown_energy'

def extract_secondary_heat_code(sec_col):
    try:
        sec_list = ast.literal_eval(sec_col)
        desc = sec_list[0]['description']
        return secondary_heat_mapping.get(desc, 'unknown_secondary')
    except:
        return 'unknown_secondary'

In [14]:
# Step 4: Apply mappings to dataset
# Description: Create the new HEATING_CODE column by combining main and secondary heating codes.

df['HEATING_CODE'] = df.apply(
    lambda row: f"{extract_main_heat_code(row['MAINHEAT_DESCRIPTION_PARSED'])}_{extract_secondary_heat_code(row['SECONDHEAT_DESCRIPTION_PARSED'])}", 
    axis=1
)

# Optional: preview new column
df[['MAINHEAT_DESCRIPTION_PARSED', 'SECONDHEAT_DESCRIPTION_PARSED', 'HEATING_CODE']].head()

,MAINHEAT_DESCRIPTION_PARSED,SECONDHEAT_DESCRIPTION_PARSED,HEATING_CODE
0,"[{'description': 'Electric storage heaters', '...","[{'description': 'Room heaters, electric', 'en...",electric_storage_heaters_good_room_heaters_ele...
1,"[{'description': 'Electric storage heaters', '...","[{'description': 'Room heaters, electric', 'en...",electric_storage_heaters_average_room_heaters_...
2,"[{'description': 'Air source heat pump, radiat...","[{'description': 'Room heaters, dual fuel (min...",air_source_heat_pump_radiators_electric_good_r...
3,"[{'description': 'Boiler and radiators, oil', ...","[{'description': 'None', 'energy_rating': 'N/A...",boiler_radiators_oil_average_none
4,"[{'description': 'Room heaters, electric', 'en...","[{'description': 'Room heaters, wood logs', 'e...",room_heaters_electric_very_poor_room_heaters_w...


In [15]:
# Step 6: Create JSON-style heating column
# Description: Adds a new column 'HEATING_JSON' that contains a list of dictionaries
# with main heat description, energy rating, and secondary heat code.

import json  # optional if you want to export later as actual JSON

def generate_heating_json(row):
    try:
        # Parse main heat
        main_list = ast.literal_eval(row['MAINHEAT_DESCRIPTION_PARSED'])
        main_desc = main_list[0]['description']
        main_energy = main_list[0]['energy_rating']
        main_code = main_heat_mapping.get(main_desc, 'unknown_main')
        energy_code = energy_rating_mapping.get(main_energy, 'unknown_energy')
        
        # Parse secondary heat
        sec_list = ast.literal_eval(row['SECONDHEAT_DESCRIPTION_PARSED'])
        sec_desc = sec_list[0]['description']
        sec_code = secondary_heat_mapping.get(sec_desc, 'unknown_secondary')
        
        # Construct JSON-style dictionary
        heating_json = [{
            'main_heat': main_code,
            'main_energy': energy_code,
            'secondary_heat': sec_code
        }]
        
        return heating_json
    except:
        return [{'main_heat': 'unknown_main', 'main_energy': 'unknown_energy', 'secondary_heat': 'unknown_secondary'}]

# Apply the function
df['HEATING_JSON'] = df.apply(generate_heating_json, axis=1)

# Optional preview
df[['HEATING_CODE', 'HEATING_JSON']].head()

,HEATING_CODE,HEATING_JSON
0,electric_storage_heaters_good_room_heaters_ele...,"[{'main_heat': 'electric_storage_heaters', 'ma..."
1,electric_storage_heaters_average_room_heaters_...,"[{'main_heat': 'electric_storage_heaters', 'ma..."
2,air_source_heat_pump_radiators_electric_good_r...,[{'main_heat': 'air_source_heat_pump_radiators...
3,boiler_radiators_oil_average_none,"[{'main_heat': 'boiler_radiators_oil', 'main_e..."
4,room_heaters_electric_very_poor_room_heaters_w...,"[{'main_heat': 'room_heaters_electric', 'main_..."


In [16]:
# Step 5: Save updated dataset
# Description: Write the DataFrame back to CSV with the new HEATING_CODE column.

output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_with_heating_code.csv"
df.to_csv(output_path, index=False)

In [18]:
# Step 7: Extract unique HEATING_CODE and HEATING_JSON combinations
# Description: Creates a new DataFrame containing only unique rows of HEATING_CODE and HEATING_JSON.

import pandas as pd

# --- Convert HEATING_JSON lists to strings for hashing ---
df['HEATING_JSON_str'] = df['HEATING_JSON'].apply(lambda x: str(x) if pd.notnull(x) else '')

# --- Select relevant columns and drop duplicates ---
unique_heating_df = df[['HEATING_CODE', 'HEATING_JSON_str']].drop_duplicates().reset_index(drop=True)

# --- Optional: Preview the first 20 unique combinations ---
print(unique_heating_df.head(20))

# --- Count total unique combinations ---
print(f"Total unique HEATING_CODE + HEATING_JSON combinations: {len(unique_heating_df)}")

# --- Save the unique combinations to CSV ---
unique_heating_df.to_csv(
    '/Users/jakegehrung/Desktop/data_raw/data_processed/unique_heating_combinations.csv',
    index=False
)

                                         HEATING_CODE  \
0   electric_storage_heaters_good_room_heaters_ele...   
1   electric_storage_heaters_average_room_heaters_...   
2   air_source_heat_pump_radiators_electric_good_r...   
3                   boiler_radiators_oil_average_none   
4   room_heaters_electric_very_poor_room_heaters_w...   
5                room_heaters_electric_very_poor_none   
6   boiler_radiators_dual_fuel_mineral_wood_averag...   
7   boiler_radiators_oil_average_room_heaters_dual...   
8   air_source_heat_pump_underfloor_electric_good_...   
9              boiler_radiators_electric_average_none   
10  electric_storage_heaters_average_room_heaters_...   
11  boiler_radiators_oil_average_room_heaters_wood...   
12  air_source_heat_pump_radiators_electric_good_r...   
13  air_source_heat_pump_radiators_electric_good_none   
14  air_source_heat_pump_radiators_electric_very_g...   
15   boiler_radiators_lpg_poor_room_heaters_wood_logs   
16  air_source_heat_pump_radiat